# L08-01　ONNX、OM 与 ATC

实验 8 的导学部分。用一个小模型走完 `训练框架模型 → ONNX → OM → 推理验证` 的全过程，说明每个环节解决什么问题。

ATC 参数的完整用法见 L08-02，msame 的验证与排错见 L08-03。

**前置**：已完成实验 2 的环境检查，了解 PyTorch 的基本用法。

**环境**：第 3、4 节只需 CPU 版 PyTorch，任意机器可跑；第 5、6 节需要昇腾开发套件与 NPU。

## 1　ONNX转换OM主要原因

训练得到的 `resnet50.pth`，拷到昇腾服务器上不能直接 `torch.load` 起来推理。原因有三层。

**指令集**。GPU 执行 CUDA 指令，昇腾执行达芬奇架构指令，两套二进制互不兼容。

**运行时**。PyTorch 的 GPU 后端调用 CUDA runtime，昇腾用 AscendCL runtime，内存管理和任务下发的接口都不同。

**关注点**。训练需要保留反向图、优化器状态和动态控制流；推理只需要前向，而且希望图是静态的，以便编译期做优化。

| | 训练态（`.pth`） | 部署态（`.om`） |
| --- | --- | --- |
| 计算图 | 动态图，随 Python 逐行构建 | 静态图，编译期确定 |
| 保存内容 | 权重 + 结构代码 + 优化器状态 | 权重 + 已编排的执行序列 |
| 运行依赖 | PyTorch 与原始模型代码 | AscendCL runtime |
| 输入 shape | 可变 | 编译期固定，或声明取值范围 |
| 图优化 | 基本不做 | 算子融合、内存复用、权重重排 |

从 PyTorch 到 OM 有三条路：经 ONNX、经 TorchScript、或换用 TensorFlow/Caffe 的导出格式。本实验走 ONNX，一是官方推荐、生态成熟，二是 ONNX 文件本身可读、可校验，出问题时能判断是导出阶段错还是编译阶段错。

## 2　全链路总览

```text
   model.pth  ──torch.onnx.export──►  model.onnx  ──atc──►  model.om
       │                                   │                    │
   torch 前向                        onnxruntime 前向        msame 推理
       ▼                                   ▼                    ▼
     输出 A ────── 对齐 ① ──────► 输出 B ────── 对齐 ② ──────► 输出 C
```

| 环节 | 角色 | 解决的问题 | 产物 |
| --- | --- | --- | --- |
| 训练框架模型 | 起点 | 模型能力的来源 | `.pth` |
| ONNX | 中间表示 | 脱离框架描述计算图 | `.onnx` |
| ATC | 编译器 | 翻译并优化成昇腾可执行形式 | 转换日志 |
| OM | 部署态模型 | 昇腾上实际加载执行的格式 | `.om` |
| 推理验证 | 质量关卡 | 确认转换没有改变模型行为 | 输出数据 |

可以类比编译过程：ONNX 相当于中间表示，ATC 相当于面向特定芯片的编译器，OM 相当于编译产物。这个类比能推出一个重要结论——**OM 与芯片型号绑定，换芯片必须重新转换**。这正是 ATC 的 `--soc_version` 必填的原因。

链路上有两个对齐点，把它们做成固定动作：相邻环节的输出对不上，问题就在这一段，不要拖到最后再查。

## 3　训练框架模型

导出 ONNX 与执行 ATC 是两件独立的内容。前者只要 CPU 版 PyTorch，任意机器都能做；后者需要昇腾开发套件。本节在前一种环境下完成。

首先安装onnx 相关python库

In [1]:
!pip install onnx onnxruntime onnxscript -i https://pypi.tuna.tsinghua.edu.cn/simple

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import torch, onnx, onnxruntime, numpy as np

print("torch       ", torch.__version__)
print("onnx        ", onnx.__version__)
print("onnxruntime ", onnxruntime.__version__)
print("providers   ", onnxruntime.get_available_providers())

torch        2.9.0+cpu
onnx         1.22.0
onnxruntime  1.28.0
providers    ['AzureExecutionProvider', 'CPUExecutionProvider']


下面定义演示模型。不需要下载权重。其中 BatchNorm 和 Dropout 是特意加的，3.3 节要用它们说明一个问题。

In [3]:
import torch.nn as nn

class DemoNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv = nn.Conv2d(3, 8, kernel_size=3, padding=1)
        self.bn = nn.BatchNorm2d(8)
        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveAvgPool2d((4, 4))
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(8 * 4 * 4, num_classes)

    def forward(self, x):
        x = self.relu(self.bn(self.conv(x)))
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        return self.fc(x)

model = DemoNet()
INPUT_SHAPE = (1, 3, 32, 32)
print(model(torch.randn(*INPUT_SHAPE)).shape)

torch.Size([1, 10])


`__init__` 里声明的层和 `forward` 里的数据流是两回事。导出 ONNX 记录的是后者——实际执行到的算子序列。`__init__` 中定义但 `forward` 未调用的层不会出现在 ONNX 里。

### 3.2　`.pth` 内部存储结构

In [4]:
import os

os.makedirs("l08_workspace", exist_ok=True)
torch.save(model.state_dict(), "l08_workspace/demo_model.pth")

sd = torch.load("l08_workspace/demo_model.pth", map_location="cpu")
for k, v in sd.items():
    print("%-24s %-16s %s" % (k, tuple(v.shape), v.dtype))

conv.weight              (8, 3, 3, 3)     torch.float32
conv.bias                (8,)             torch.float32
bn.weight                (8,)             torch.float32
bn.bias                  (8,)             torch.float32
bn.running_mean          (8,)             torch.float32
bn.running_var           (8,)             torch.float32
bn.num_batches_tracked   ()               torch.int64
fc.weight                (10, 128)        torch.float32
fc.bias                  (10,)            torch.float32


列出来的全是权重张量和 BatchNorm 的统计量，没有任何一项描述这些层如何连接。所以 `torch.load` 之前必须先有 `DemoNet` 的类定义——这就是"依赖原始代码"的具体含义。

ONNX 把结构和权重打包在同一个文件里，这是它能跨框架的前提。

### 3.3　导出前必须 `model.eval()`

In [5]:
torch.manual_seed(0)
x = torch.randn(*INPUT_SHAPE)

model.train()
with torch.no_grad():
    a1, a2 = model(x), model(x)

model.eval()
with torch.no_grad():
    b1, b2 = model(x), model(x)

print("train 模式两次前向的最大差异: %.6f" % (a1 - a2).abs().max())
print("eval  模式两次前向的最大差异: %.6f" % (b1 - b2).abs().max())
print("两种模式之间的最大差异     : %.6f" % (a1 - b1).abs().max())

train 模式两次前向的最大差异: 0.530249
eval  模式两次前向的最大差异: 0.000000
两种模式之间的最大差异     : 0.647967


train 模式下同一输入两次前向结果不同：Dropout 每次随机置零的位置不一样，BatchNorm 还会更新 running stats。eval 模式下 Dropout 关闭、BatchNorm 用固定统计量，结果可复现。

导出前忘记 `eval()`，ONNX 会记录下 Dropout 的随机行为和当次的 BN 状态。导出过程不报错，但结果与预期不符，且现象隐蔽。

**`torch.onnx.export` 之前一定先 `model.eval()`**。

## 4   导出 ONNX

ONNX（Open Neural Network Exchange）是开放的模型交换格式。一个 ONNX 模型包含四部分：网络结构、权重参数、输入输出信息、算子信息。

### 4.1　导出的机制

PyTorch 是动态图，ONNX 是静态图，中间的落差靠 trace 弥合。给模型喂一个 shape 正确的输入跑一次前向，记录实际走过的算子——这就是 `torch.onnx.export` 需要 `dummy_input` 的原因。输入的数值无所谓，shape 和 dtype 对就行。

trace 有固有局限，都源于"只记录本次执行":

- `if` 只记录这次走到的分支，另一支永久丢失
- `for` 的循环次数被固化成常量
- Python 标量和 numpy 值被当作常量写死

需要保留控制流时用 `torch.jit.script`，它做真正的语法分析。

### 4.2　最小可用的导出

In [6]:
model.eval()
dummy_input = torch.randn(*INPUT_SHAPE)

torch.onnx.export(model, dummy_input, "l08_workspace/tmp.onnx")

m = onnx.load("l08_workspace/tmp.onnx")
print("输入名:", [i.name for i in m.graph.input])
print("输出名:", [o.name for o in m.graph.output])

[torch.onnx] Obtain model graph for `DemoNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `DemoNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
输入名: ['x']
输出名: ['linear']


一个可选参数都不给也能导出成功，但输入输出的名字是自动生成的，模型稍有改动就可能变化。而下游的 ATC `--input_shape` 和 msame 都靠名字定位输入，名字一变命令全部失效。所以要显式命名。

### 4.3　参数说明

| 参数 | 作用 | 说明 |
| --- | --- | --- |
| `model` | 待导出模型 | 必须已 `eval()` |
| `args` | 示例输入 | 只需 shape 与 dtype 正确；多输入传 tuple |
| `f` | 输出路径 | |
| `input_names` / `output_names` | 节点命名 | 显式指定，下游靠它对齐 |
| `opset_version` | 算子集版本 | 过低会缺算子；需在 CANN 支持范围内 |
| `dynamic_axes` | 声明可变维度 | 不写则 shape 全部固定 |
| `do_constant_folding` | 常量折叠 | 默认 True，提前算掉常量子图 |
| `export_params` | 是否带权重 | 默认 True |

`input_names` 是贯穿全链路的接口契约，改一次名下游所有命令都要跟着改。

In [7]:
torch.onnx.export(
    model,
    dummy_input,
    "l08_workspace/demo_model.onnx",
    input_names=["input"],
    output_names=["output"],
    opset_version=18,
    do_constant_folding=True,
)

print("%.1f KB" % (os.path.getsize("l08_workspace/demo_model.onnx") / 1024))

[torch.onnx] Obtain model graph for `DemoNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `DemoNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
6.0 KB


### 4.4　动态维度

不写 `dynamic_axes` 时所有维度都固定成 `dummy_input` 的 shape。要让某一维可变，就在这里声明：

```python
dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}}   # 推荐：给维度命名
dynamic_axes={"input": [0], "output": [0]}                     # 只给序号，报错信息里没有名字
```

动态 shape 换来灵活性，代价是编译器可优化的空间变小，且下游命令更复杂。这条链路上的选择是一整套的：

| 场景 | ONNX 导出 | ATC | msame |
| --- | --- | --- | --- |
| 静态 | 不设 `dynamic_axes` | `--input_shape` | 无需额外参数 |
| 动态 batch | `{0: "batch"}` | `--dynamic_batch_size` | `--dymBatch` |
| 动态分辨率 | `{2: "h", 3: "w"}` | `--dynamic_image_size` | 相应指定 |
| 完全动态 | 多维声明 | `--input_shape_range` | `--dymShape` + `--outputSize` |

本节主线用静态 shape，动态场景在 L08-02、L08-03 展开。

### 4.5　导出阶段的常见问题

这些问题的共同特征是**导出不报错，但结果不对**：

- 用了 `tensor.item()`、`tensor.data`，张量被降级成常量
- dict 或 str 作输入会被当常量，作输出会被丢弃；输入输出应为 tensor
- 嵌套的 list/tuple 会被展开
- 对张量做 inplace 操作，可能导致导出结果与 PyTorch 不一致
- `nn.Upsample` 等算子兼容性较差，可提高 `opset_version` 或改用等价写法

正因为这些问题不会在导出时暴露，才需要下一节的验证。

## 5　ONNX 模型体检


需要检查的三个内容：结构是否合法、图是什么样、数值是否对得上。

In [8]:
m = onnx.load("l08_workspace/demo_model.onnx")
onnx.checker.check_model(m)

print("IR version   :", m.ir_version)
print("opset        :", [(op.domain or "ai.onnx", op.version) for op in m.opset_import])
print("producer     :", m.producer_name, m.producer_version)

IR version   : 10
opset        : [('ai.onnx', 18)]
producer     : pytorch 2.9.0+cpu


`check_model` 只检查格式合法性，不检查计算结果。通过它不代表模型没问题。

In [9]:
from collections import Counter

def describe(vi):
    t = vi.type.tensor_type
    dims = [d.dim_param if d.dim_param else d.dim_value for d in t.shape.dim]
    return "%-8s %-8s %s" % (vi.name, onnx.TensorProto.DataType.Name(t.elem_type), dims)

for i in m.graph.input:
    print("输入", describe(i))
for o in m.graph.output:
    print("输出", describe(o))

print("\n算子:", dict(Counter(n.op_type for n in m.graph.node)))

输入 input    FLOAT    [1, 3, 32, 32]
输出 output   FLOAT    [1, 10]

算子: {'Conv': 1, 'Relu': 1, 'AveragePool': 1, 'Reshape': 1, 'Gemm': 1}


三点要确认：名字与 `input_names` 一致；shape 里出现字符串（如 `batch`）说明是动态维；算子清单是判断 CANN 是否支持的第一手依据。

图结构的可视化可以用 Netron，比逐个打印节点直观。

In [10]:
import onnxruntime as ort

sess_options = ort.SessionOptions()


sess_options.intra_op_num_threads = 1
sess_options.inter_op_num_threads = 1

sess = ort.InferenceSession(
    "l08_workspace/demo_model.onnx",
    sess_options=sess_options,
    providers=["CPUExecutionProvider"]
)

np.random.seed(0)
x_np = np.random.randn(*INPUT_SHAPE).astype(np.float32)

ort_out = sess.run(
    None,
    {"input": x_np}
)[0]

print("输出 shape:", ort_out.shape)
print(np.round(ort_out[0][:5], 6))

输出 shape: (1, 10)
[-0.054915  0.206852 -0.152516 -0.180631  0.278577]


输入 dtype 必须是 `np.float32`，与 torch 的 float32 对应，写成 float64 会直接报类型错误。输入字典的 key 就是导出时的 `input_names`；如果模型不是自己导出的，用 `sess.get_inputs()[0].name` 取。

`run` 的第一个参数指定要哪些输出，传 `None` 表示按顺序全部返回。

### 对齐 ①：torch 与 onnxruntime

In [11]:
model.eval()
with torch.no_grad():
    torch_out = model(torch.from_numpy(x_np)).numpy()

print("最大绝对误差 %.3e" % np.abs(torch_out - ort_out).max())
np.testing.assert_allclose(torch_out, ort_out, rtol=1e-3, atol=1e-5)
print("对齐 ① 通过")

最大绝对误差 1.490e-07
对齐 ① 通过


不要求完全相等：两边的算子实现和计算顺序不同，浮点累加的舍入误差不可避免。`rtol=1e-3, atol=1e-5` 是常用容差。

这一步不通过就不要往下走，回到 4.5 的清单排查，重点看 `eval()` 和 inplace 操作。

## 6　第三站：ATC 转 OM

ATC（Ascend Tensor Compiler）是 CANN 的模型转换工具，把 ONNX 等格式的模型编译成昇腾可执行的 OM 离线模型。"离线"指转换在部署前一次性完成，不占用线上推理时间。

转换过程内部依次做：模型解析 → shape 推导 → 算子融合 → 内存规划 → 图优化 → 生成 OM。

以下命令需要昇腾开发套件。

### 6.1　环境变量

`atc: command not found` 绝大多数情况是没有 source 环境脚本。新版 CANN 只需一行：

In [12]:
!source /usr/local/Ascend/ascend-toolkit/set_env.sh && atc --help | head -20

ATC start working now, please wait for a moment.
usage: atc <args>
generate offline model example:
atc --model=./alexnet.prototxt --weight=./alexnet.caffemodel --framework=0 --output=./domi --soc_version=<soc_version> 
generate offline model for single op example:
atc --singleop=./op_list.json --output=./op_model --soc_version=<soc_version> 

===== Basic Functionality =====
[General]
  --h/help            Show this help message
  --mode              Run mode.
                       0: default, generate offline model;
                       1: convert model to JSON format;
                       3: only pre-check;
                       5: convert ge dump txt file to JSON format;
                       6: display model info;
                       30: convert original graph to execute-om for nano(offline model)

[Input]
  --model             Model file


注意每个 `%%bash` 或 `!` 都是独立的子进程，`source` 的结果不会带到下一个 cell。所以后面每个用到 `atc` 的 cell 都要重新 source。要让它长期生效，把这行写进 `~/.bashrc`。

安装 CANN 时另有两点需要注意：`umask` 必须是 `0022`，否则安装后文件权限异常；开发环境与运行环境架构不同时（如 x86 开发、aarch64 部署），开发环境要同时安装两种架构的 toolkit。完整步骤见官方文档。

### 6.2　确定 soc_version

In [13]:
!npu-smi info

+------------------------------------------------------------------------------------------------+
| npu-smi 25.5.1                   Version: 25.5.1                                               |
+---------------------------+---------------+----------------------------------------------------+
| NPU   Name                | Health        | Power(W)    Temp(C)           Hugepages-Usage(page)|
| Chip                      | Bus-Id        | AICore(%)   Memory-Usage(MB)  HBM-Usage(MB)        |
+===========================+===============+====================================================+
| 6     910B4               | OK            | 97.9        42                0    / 0             |
| 0                         | 0000:82:00.0  | 0           0    / 0          2868 / 32768         |
+===========================+===============+====================================================+
+---------------------------+---------------+----------------------------------------------------+
| NPU     

确认硬件环境，除了 `npu-smi info`，也可以用 ACL 接口精确查询：

In [14]:
import acl
acl.init()
print(acl.get_soc_name())

Ascend910B4


### 6.3　执行转换

把下面的 `--soc_version` 替换为上一步的实测值。

In [15]:
%%bash
source /usr/local/Ascend/ascend-toolkit/set_env.sh
export TE_PARALLEL_COMPILER=1
export MAX_COMPILE_PROCESS_NUM=1

atc --model=l08_workspace/demo_model.onnx \
    --framework=5 \
    --output=l08_workspace/demo_model \
    --input_format=NCHW \
    --input_shape="input:1,3,32,32" \
    --soc_version=Ascend910B4 \
    --log=info

ATC start working now, please wait for a moment.
..

/usr/local/python3.11.14/lib/python3.11/site-packages/torch_npu/utils/collect_env.py:58: UserWarning: Warning: The /usr/local/Ascend/cann-8.5.0 owner does not match the current owner.
  warnings.warn(f"Warning: The {path} owner does not match the current owner.")
/usr/local/python3.11.14/lib/python3.11/site-packages/torch_npu/utils/collect_env.py:58: UserWarning: Warning: The /usr/local/Ascend/cann-8.5.0/aarch64-linux/ascend_ops_install.info owner does not match the current owner.
  warnings.warn(f"Warning: The {path} owner does not match the current owner.")
/usr/local/python3.11.14/lib/python3.11/site-packages/torch_npu/utils/_path_manager.py:66: UserWarning: Permission mismatch: The owner of /usr/local/python3.11.14/lib/python3.11/site-packages/torch_npu/lib/libop_plugin_atb.so does not match.
  warnings.warn(f"Permission mismatch: The owner of {path} does not match.")
/usr/local/python3.11.14/lib/python3.11/site-packages/torch_npu/utils/collect_env.py:58: UserWarning: Warning: The 

path string is NULLpath string is NULL...
ATC run success, welcome to the next use.



转换成功后会生成 `model.om` 以及元信息 JSON。报错时优先看日志里 `[ERROR]` 行：不支持的算子会点名、shape 不匹配会指出节点。

静态 shape 场景下使用 `--input_shape`，格式为 `input_name:N,C,H,W`。多个输入用分号隔开：

```bash
--input_shape="input1:1,3,224,224;input2:1,100"
```

动态 shape 的参数在 L08-02 说明。

### 6.4　ATC 常用参数

| 参数 | 说明 |
| --- | --- |
| `--model` | 输入模型路径 |
| `--framework` | 框架类型；5=ONNX |
| `--output` | 输出路径（不含扩展名）；.om 自动补 |
| `--soc_version` | 目标芯片型号；必须与运行环境一致 |
| `--input_shape` | 静态场景：显式指定输入 shape |
| `--dynamic_batch_size` | batch 可变；逗号分隔支持值 |
| `--dynamic_image_size` | 分辨率可变 |
| `--input_shape_range` | 完全动态范围 |
| `--log=level` | 日志级别；debug/info/warning/error |
| `--precision_mode` | 精度模式；allow_fp32_to_fp16、force_fp16 等 |
| `--fusion_switch_file` | 算子融合开关配置（调试用） |

更多参数见 CANN 开发工具文档。

## 7　第四站：msame 推理验证

### 7.1　获取 msame

从 https://gitee.com/ascend/tools/tree/master/msame 获取源码。


OM 里是优化后的计算图、重排后的权重，以及针对目标芯片的执行调度信息。三个特点：

1. 二进制格式，不可读，无法还原成 ONNX
2. 与芯片型号强绑定，换型号必须重新转换
3. 运行时只依赖 AscendCL，不再需要 PyTorch

与 ONNX 的体积差异来自权重重排、精度模式和附加的调度信息，方向不固定，不要假设一定变小。

克隆后编译：

In [17]:
x_np.tofile("l08_workspace/input.bin")
print("字节数", os.path.getsize("l08_workspace/input.bin"))
print("核算  ", np.prod(INPUT_SHAPE), "元素 x 4 字节 =", np.prod(INPUT_SHAPE) * 4)

字节数 12288
核算   3072 元素 x 4 字节 = 12288


编译成功后二进制在 `out/msame`。

### 7.2　准备输入文件

msame 要求输入为 `.bin` 格式的原始二进制，一个文件对应一个输入。静态 shape 时连 shape 都不必显式传，从 OM 元信息自动读取。

In [19]:

!/opt/atomgit/tools/msame/out/msame --model l08_workspace/demo_model.om \
      --input l08_workspace/input.bin \
      --output l08_workspace/msame_out \
      --outfmt BIN

[INFO] acl init success
[INFO] open device 0 success
[INFO] create context success
[INFO] create stream success
[INFO] get run mode success
[INFO] load model l08_workspace/demo_model.om success
[INFO] create model description success
[INFO] get input dynamic gear count success
[INFO] create model output success
l08_workspace/msame_out/2026814_15_43_20_91581
[INFO] start to process file:l08_workspace/input.bin
[INFO] model execute success
Inference time: 0.222ms
[INFO] get max dynamic batch size success
[INFO] output data success
Inference average time: 0.222000 ms
[INFO] destroy model input success
[INFO] unload model success, model Id is 1
[INFO] pid: 4311 Execute sample success
[INFO] end to destroy stream
[INFO] end to destroy context
[INFO] end to reset device is 0
[INFO] end to finalize acl


`--input` 可以给单个 bin，也可以给目录批量推理。`--output` 下面 msame 会按时间戳建子目录。`--outfmt` 取 `BIN` 或 `TXT`，调试时用 TXT 肉眼看，比对时用 BIN。

### 对齐 ②：onnxruntime 与 msame

In [ ]:
import glob

out_bin = sorted(glob.glob("l08_workspace/msame_out/*/output_0.bin"))[-1]
msame_out = np.fromfile(out_bin, dtype=np.float32).reshape(1, 10)

print("最大绝对误差 %.3e" % np.abs(ort_out - msame_out).max())
print("argmax 一致?", (ort_out.argmax() == msame_out.argmax()))

cos_sim = (ort_out * msame_out).sum() / (np.linalg.norm(ort_out) * np.linalg.norm(msame_out))
print("余弦相似度 %.6f" % cos_sim)

np.testing.assert_allclose(ort_out, msame_out, rtol=1e-2, atol=1e-3)
print("对齐 ② 通过")

这里的容差要比对齐 ① 放宽。默认精度模式下计算用 FP16，单次运算的相对误差约 1e-4，但误差会逐层累积，深网络上可能达到 1e-2 甚至更大。

判定分三级：

1. **argmax 是否一致**——分类任务最直接的判据
2. **余弦相似度**——整体分布是否接近
3. **逐元素容差**——用于定位具体哪一层出问题

只有第三级不通过、前两级通过，通常是精度模式导致的正常现象；如果 argmax 都变了，说明转换确实有问题。

关于耗时：首次推理包含模型加载、内存分配和算子初始化，必须与后续推理分开统计。用 `--loop` 跑多次，丢掉前几次再取平均。

成功后输出保存在 `./out/<timestamp>_output_0.bin`。如果有多个输出，后缀会是 `_1`、`_2`。报错时重点看 ACL error code，对照 CANN 文档排查。

### 7.4　msame 常用参数

| 参数 | 说明 |
| --- | --- |
| `--model` | OM 模型路径 |
| `--input` | 输入文件或目录；bin 格式 |
| `--output` | 输出目录 |
| `--device` | 设备 ID，默认 0 |
| `--dymBatch` / `--dymHW` / `--dymShape` | 动态场景下指定实际 shape |
| `--outputSize` | 动态场景下需预分配输出 buffer 大小 |
| `--loop` | 重复推理次数，用于性能测试 |
| `--debug` | 打印更多日志 |

静态 shape 时 `--input` 指向一个或多个 bin，不用手动指定 shape。动态场景会复杂些，L08-02 展开。



## 9　练习

1. 把演示模型改成动态 batch 重新导出，用 5.2 的代码观察输入 shape 的变化。
2. 去掉 `model.eval()` 重新导出，运行对齐 ①，记录现象并解释原因。
3. 把 `--input_shape` 里的输入名故意写错，记录 ATC 的完整报错。
4. 对比 ONNX 与 OM 的文件大小，结合精度模式解释差异来源。
5. 如果模型里有一个 `if` 分支依赖输入的数值，trace 导出会发生什么？应该怎么处理？

提交：导出脚本、ATC 命令与完整日志、两处对齐结果、第 2 和第 3 题的现象记录。注明硬件型号与 CANN 版本，不要编造耗时和精度数据。